In [1]:
import pandas as pd
import torch
import pytorch_lightning
from typing import Dict, Any, List, Optional
from dgl import batch, DGLGraph
from dgllife.utils import smiles_to_bigraph, CanonicalAtomFeaturizer, CanonicalBondFeaturizer
from dgllife.model import MPNNPredictor
import toml
from pathlib import Path

In [2]:
# Atom and Bond Featurizers
NF = CanonicalAtomFeaturizer()
BF = CanonicalBondFeaturizer()

In [3]:
class DistanceNetworkLigthning(pytorch_lightning.LightningModule):

    def __init__(self, embed_size, node_in_feats, edge_in_feats):
        super().__init__()
        self.save_hyperparameters()
        self._net = MPNNPredictor(node_in_feats=node_in_feats,
                                  edge_in_feats=edge_in_feats,
                                  n_tasks=embed_size)
        self._loss = torch.nn.TripletMarginLoss(margin=1.0, p=2)
        return None

    def forward(self, anchor, positive, negative):
        anchor_out = self._net(*anchor)
        positive_out = self._net(*positive)
        negative_out = self._net(*negative)
        return anchor_out, positive_out, negative_out

    def training_step(self, batch, batch_nb):
        anchor, positive, negative = batch
        anchor_out, positive_out, negative_out = self.forward(anchor, positive, negative)
        loss = self._loss(anchor_out, positive_out, negative_out)
        self.log('train_loss', loss, on_step=True, on_epoch=False)
        return loss

    def validation_step(self, batch, batch_nb):
        anchor, positive, negative = batch
        anchor_out, positive_out, negative_out = self.forward(anchor, positive, negative)
        loss = self._loss(anchor_out, positive_out, negative_out)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def test_step(self, batch, batch_nb): 
        anchor, positive, negative = batch
        anchor_out, positive_out, negative_out = self.forward(anchor, positive, negative)
        loss = self._loss(anchor_out, positive_out, negative_out)
        self.log('test_loss', loss, sync_dist=True)
        return loss

    def configure_optimizers(self):  
        opt = torch.optim.Adam(self.parameters(), lr=1.0e-3)
        return opt


In [4]:
def load_model(config: Dict[str, Any]) -> torch.nn.Module:
    """
    Loads and initializes the model with the specified parameters from the configuration.

    Args:
        config (Dict[str, Any]): Dictionary with model configuration, including model path and parameters.

    Returns:
        torch.nn.Module: The initialized and loaded model, ready for evaluation.
    """
    model = DistanceNetworkLigthning(**config['generate_chemdist']['params'])

    state_dict = torch.load(config['generate_chemdist']["model_path"])
    if 'state_dict' in state_dict:
        model.load_state_dict(state_dict['state_dict'])
    else:
        model.load_state_dict(state_dict)

    model.eval()
    #model.cuda()
    return model

In [5]:
def load_feature_config(config_path: str) -> Dict[str, Any]:
    """
    Reads the configuration from a TOML file and returns parameters as a dictionary.

    Args:
        config_path (str): Path to the TOML configuration file.

    Returns:
        Dict[str, Any]: Dictionary containing configuration parameters.
    """
    config = toml.load(config_path)

    # Check if at least one descriptor type is set to true
    if not any([config.get("morgan", False), config.get("maccs_keys", False), config.get("chemdist", False)]):
        warnings.warn(
            "No descriptor types are enabled in the configuration. Ensure at least one of 'morgan', 'maccs_keys', or 'chemdist' is set to true.")

    return {
        "input_path": Path(config.get("input_path", "")),
        "output_path": config.get("output_path", ""),
        "file_pattern": config.get("file_pattern", "*.smi"),
        "chemdist_path": config.get("chemdist_path", ""),
        "chemdist_params": config.get("chemdist_params", {}),
        "generate_chemdist": config.get("chemdist", True),
    }


In [6]:
config_path = load_feature_config('./features.toml')

In [7]:
config_path

{'input_path': PosixPath('.'),
 'output_path': '',
 'file_pattern': '*.smi',
 'chemdist_path': '',
 'chemdist_params': {},
 'generate_chemdist': {'model_path': './model_trained.pt',
  'device': 'cuda',
  'params': {'edge_in_feats': 12, 'embed_size': 16, 'node_in_feats': 74}}}

In [8]:
config_path['generate_chemdist']['model_path']

'./model_trained.pt'

In [9]:
config_path['generate_chemdist']['params']

{'edge_in_feats': 12, 'embed_size': 16, 'node_in_feats': 74}

In [10]:
model = load_model(config_path)

/tmp/ipykernel_50982/527696548.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(config['generate_chemdist']["model_path"])


In [11]:
model.print

<bound method LightningModule.print of DistanceNetworkLigthning(
  (_net): MPNNPredictor(
    (gnn): MPNNGNN(
      (project_node_feats): Sequential(
        (0): Linear(in_features=74, out_features=64, bias=True)
        (1): ReLU()
      )
      (gnn_layer): NNConv(
        (edge_func): Sequential(
          (0): Linear(in_features=12, out_features=128, bias=True)
          (1): ReLU()
          (2): Linear(in_features=128, out_features=4096, bias=True)
        )
      )
      (gru): GRU(64, 64)
    )
    (readout): Set2Set(
      n_iters=6
      (lstm): LSTM(128, 64, num_layers=3)
    )
    (predict): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=16, bias=True)
    )
  )
  (_loss): TripletMarginLoss()
)>

In [12]:
df1 = pd.read_csv('./final_total.csv')
df1

,CHEMBLID,smiles,ido_ic50,tdo_ic50
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,23.0,60.0
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,10000.0,10000.0
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,183.0,224200.0
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,11790.0,12250.0
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,14610.0,30230.0
...,...,...,...,...
755,CHEMBL5221020,O[C@@H]1CCCC[C@H]1CNc1c(Br)ccc2[nH]ncc12,1230.0,11240.0
756,CHEMBL5221102,Brc1cc(NC[C@H]2CCCN2)c2cn[nH]c2c1,56450.0,7450.0
757,CHEMBL578036,CC1(C)CCCc2c1ccc1c2C(=O)C(=O)c2c(CO)coc2-1,10000.0,450.0
758,CHEMBL584991,Nc1nonc1/C(=N/O)Nc1ccc(F)c(Cl)c1,50.0,10000.0


In [13]:
df1['smiles']

0                O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O
1      Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C
2               O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O
3           Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1
4                   O=[N+]([O-])c1cc(F)c2cccnc2c1O
                          ...                     
755       O[C@@H]1CCCC[C@H]1CNc1c(Br)ccc2[nH]ncc12
756              Brc1cc(NC[C@H]2CCCN2)c2cn[nH]c2c1
757     CC1(C)CCCc2c1ccc1c2C(=O)C(=O)c2c(CO)coc2-1
758               Nc1nonc1/C(=N/O)Nc1ccc(F)c(Cl)c1
759             O=C1c2cc(Br)ccc2-n2c1nc1ccccc1c2=O
Name: smiles, Length: 760, dtype: object

In [14]:
def chemdist_func_batch(graph_list, model, NF, BF):
    """
    Process a batch of molecular graphs and generate embeddings using the provided model.
    """
    bg = batch(graph_list)
    bg = bg.to('cpu')
    nfeats = bg.ndata.pop('h')
    efeats = bg.edata.pop('e')

    with torch.no_grad():
        output = model._net(bg, nfeats, efeats).cpu().detach().numpy()

    del bg
    del nfeats
    del efeats
    torch.cuda.empty_cache()

    return output

In [15]:
def generate_embeddings(df: pd.DataFrame, model: torch.nn.Module,
                        node_featurizer: CanonicalAtomFeaturizer,
                        edge_featurizer: CanonicalBondFeaturizer) -> pd.DataFrame:
    """
    Generates graph-based embeddings for molecules in the DataFrame, stores them in a new column,
    and removes rows with NaN embeddings.

    Args:
        df (pd.DataFrame): DataFrame containing a 'smi' column with SMILES strings.
        model (torch.nn.Module): Pre-trained model to generate embeddings.
        node_featurizer (CanonicalAtomFeaturizer): Featurizer for atoms in the molecular graph.
        edge_featurizer (CanonicalBondFeaturizer): Featurizer for bonds in the molecular graph.

    Returns:
        pd.DataFrame: Updated DataFrame with a new 'embed' column containing embeddings, and rows
                      with NaN values in the 'embed' column removed.
    """
    # Convert SMILES strings to molecular graphs
    graph_list = df['smiles'].apply(lambda x: smiles_to_bigraph(smiles=x,
                                                             node_featurizer=node_featurizer,
                                                             edge_featurizer=edge_featurizer))

    # Generate embeddings in batches
    embeddings = chemdist_func_batch(graph_list.tolist(), model, node_featurizer, edge_featurizer)

    # Add embeddings to DataFrame and remove rows with NaN embeddings
    df1['embed'] = pd.Series(list(embeddings))
    #df = df.dropna(subset=['embed'])
    #df = remove_duplicate_rows(df, 'embed')     it was deleted!!!!

    return df

In [16]:
df2 = generate_embeddings(df1, model, NF, BF)

In [17]:
df2

,CHEMBLID,smiles,ido_ic50,tdo_ic50,embed
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,23.0,60.0,"[-14.003065, -23.058441, 46.13716, -7.5264153,..."
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,10000.0,10000.0,"[-8.993788, -30.84298, 37.763935, -12.150796, ..."
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,183.0,224200.0,"[-13.749965, -23.094172, 46.29172, -7.6663136,..."
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,11790.0,12250.0,"[-14.3434515, -20.237873, 46.55186, -10.75259,..."
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,14610.0,30230.0,"[-14.600085, -24.964235, 47.840824, -9.83812, ..."
...,...,...,...,...,...
755,CHEMBL5221020,O[C@@H]1CCCC[C@H]1CNc1c(Br)ccc2[nH]ncc12,1230.0,11240.0,"[-6.6100674, -35.240223, 37.34721, -13.406092,..."
756,CHEMBL5221102,Brc1cc(NC[C@H]2CCCN2)c2cn[nH]c2c1,56450.0,7450.0,"[-10.37818, -32.02663, 39.73027, -14.188325, -..."
757,CHEMBL578036,CC1(C)CCCc2c1ccc1c2C(=O)C(=O)c2c(CO)coc2-1,10000.0,450.0,"[-9.41469, -32.008568, 38.02041, -12.887679, -..."
758,CHEMBL584991,Nc1nonc1/C(=N/O)Nc1ccc(F)c(Cl)c1,50.0,10000.0,"[-14.0334, -24.151228, 44.50949, -13.630947, -..."


In [30]:
df2.to_csv('testdata_ChemDist.csv')